# Lab 5 · Mapping forest fire

**Day 3 · about 40 minutes · Student notebook**

> **Goal.** Map the Chiang Mai burn season three ways — active fire, burned area, and burn severity — and catch a 24× error that looks completely convincing.

---

### How to work in this notebook

Each step gives you a **prompt card**. Copy it into Claude (or Gemini in Colab),
paste the code you get back into the empty cell below it, and run it.
Then read the **check** underneath and make sure your numbers are believable.

If the AI's code throws an error: copy the **whole** error message, paste it back to the
assistant with the sentence *"this is the error I got, please fix the code"*, and try again.
Do not retype the code by hand.


## Three different questions about fire

| Question | Dataset | What it actually tells you |
|---|---|---|
| *Where is it burning right now?* | `FIRMS` | a hot pixel at the moment of overpass |
| *How many hectares burned?* | `MODIS/061/MCD64A1` | the burn scar, mapped monthly |
| *How badly did it burn?* | dNBR from Sentinel-2 | severity, computed by you |

Using the wrong one for the question is the most common mistake in fire mapping. Active fire
detections are **not** burned area — a single fire burning for a week gets counted many times,
and a fire that burns between overpasses gets counted not at all.

### Setup

In [ ]:
# Run this first, every session. Colab forgets everything when it recycles.
!pip install -q geemap

import ee, geemap

ee.Authenticate()                      # opens a link - sign in, paste the code back
ee.Initialize(project='YOUR-PROJECT-ID')   # <-- put YOUR project ID here

print('Earth Engine is ready.')

In [ ]:
PROVINCE = 'Chiang Mai'     # fire season here runs roughly February to April

aoi_fc = (ee.FeatureCollection('FAO/GAUL/2015/level1')
          .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
          .filter(ee.Filter.eq('ADM1_NAME', PROVINCE)))
aoi = aoi_fc.geometry()
print(f'{PROVINCE}: {aoi.area(1000).divide(1e4).getInfo():,.0f} ha')

In [ ]:
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

def mask_s2(img):
    """Drop cloud, shadow and cirrus pixels using the SCL band, then scale to reflectance."""
    scl = img.select('SCL')
    good = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(good).divide(10000)

def s2_composite(start, end, max_cloud=30):
    """Median composite over a date range - the standard way to get a cloud-free picture."""
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud))
            .map(mask_s2)
            .median()
            .select(BANDS))

### Step 1 — active fire detections

In [ ]:
YEAR = 2024

firms = (ee.ImageCollection('FIRMS')
         .filterDate(f'{YEAR}-01-01', f'{YEAR}-05-01')
         .filterBounds(aoi).select('T21').max().clip(aoi))

Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.addLayer(firms, {'min': 300, 'max': 400,
                     'palette': ['#FBF0EA', '#E8B9A4', '#D97757', '#8C2F1B']},
             f'FIRMS active fire {YEAR}')
Map.addLayer(aoi_fc.style(color='191917', fillColor='00000000', width=2), {}, 'Boundary')

n = firms.gt(0).reduceRegion(ee.Reducer.sum(), aoi, 1000,
                             maxPixels=1e9, bestEffort=True).getInfo()['T21']
print(f'{int(n):,} one-kilometre pixels recorded at least one fire detection')
Map

### Step 2 — how much actually burned

#### 🤖 Prompt card — Burned area for three seasons

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
PURPOSE   Find how many hectares burned in my province in each of the last three fire seasons.
RESOURCE  Use MODIS/061/MCD64A1, band BurnDate (0 = unburned, 1-366 = day of year).
OUTLINE   Region is `aoi`. For each year 2023, 2024, 2025, use dates January 1 to June 1.
MUST      Take the max of BurnDate over the season, treat > 0 as burned,
          multiply by pixel area in hectares, and sum. Use scale=500.
PLATFORM  Google Colab, Earth Engine Python API.
TEST      Print one line per year so I can compare the three seasons.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


> ### ✓ Check your answer
>
> Chiang Mai: **2023 → 73,039 ha**, **2024 → 67,128 ha**, **2025 → 33,482 ha**.
> 
> 2025 was a much lighter season, roughly half of 2024. Notice that the FIRMS detection counts tell
> the same story (6,409 → 6,494 → 3,311 pixels). **Two independent sensors agreeing is the best
> evidence you will get** that a trend is real and not an artefact.

### Step 3 — how badly it burned, and a trap

Burn severity uses the **Normalised Burn Ratio**, NBR = (NIR − SWIR2) / (NIR + SWIR2).
Fire kills vegetation and dries the soil, which lowers NIR and raises SWIR — so NBR drops.

**dNBR = NBR_before − NBR_after.** Bigger drop, more severe burn.

Let us compute it the obvious way first.

#### 🤖 Prompt card — Burn severity with dNBR

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
PURPOSE   Map how severely the forest burned during the 2024 fire season.
RESOURCE  Use COPERNICUS/S2_SR_HARMONIZED. A helper `s2_composite(start, end, max_cloud)`
          already exists and returns a cloud-masked median composite.
OUTLINE   Before the season: 2023-12-01 to 2024-01-31. After: 2024-04-01 to 2024-05-15.
          Area is `aoi`.
MUST      NBR = normalizedDifference of B8 and B12.
          dNBR = NBR_before - NBR_after, multiplied by 1000.
          Classify with the USGS thresholds: below 100 unburned, 100-270 low,
          270-440 moderate-low, 440-660 moderate-high, above 660 high.
          Report the hectares in each severity class using scale=100.
PLATFORM  Google Colab, Earth Engine Python API.
TEST      Print the total burned hectares so I can compare it to the MODIS figure.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


> ### ✓ Check your answer
>
> ### Stop. Your dNBR map says 1,594,000 ha burned. MODIS says 67,128 ha.
> 
> That is **24 times too much** — more than half the entire province supposedly burned. The map
> will look completely convincing. It is wrong.
> 
> **Why?** Northern Thailand is full of **deciduous forest**. Between December and April those
> trees drop their leaves. Leaf-off looks spectrally almost identical to burnt: NIR falls, SWIR
> rises, NBR drops. dNBR cannot tell "this forest burned" from "this forest is having a normal dry
> season".
> 
> **The fix:** only compute severity *inside a burn scar that an independent product confirms*.
> dNBR is good at answering **how badly**, and bad at answering **whether**. So let MCD64A1 answer
> "whether", and dNBR answer "how badly".

### Step 4 — the corrected severity map

In [ ]:
burn_scar = (ee.ImageCollection('MODIS/061/MCD64A1')
             .filterDate(f'{YEAR}-01-01', f'{YEAR}-06-01')
             .select('BurnDate').max().gt(0))

sev_masked = severity(dnbr).updateMask(burn_scar).clip(aoi)

Map.addLayer(sev_masked, {'min': 1, 'max': 5,
                          'palette': [SEV[i][1] for i in sorted(SEV)]}, 'Burn severity')
Map.add_legend(title='Burn severity', legend_dict={v[0]: v[1] for v in SEV.values()})

a2 = (ee.Image.pixelArea().divide(1e4).addBands(sev_masked)
      .reduceRegion(ee.Reducer.sum().group(groupField=1, groupName='sev'),
                    aoi, 100, maxPixels=1e9, bestEffort=True).getInfo())

tot = 0
print(f'Burn severity within the confirmed {YEAR} burn scar')
print('-' * 34)
for g in sorted(a2['groups'], key=lambda x: x['sev']):
    if g['sev'] in SEV:
        print(f"  {SEV[g['sev']][0]:<15}{g['sum']:>14,.0f} ha")
        tot += g['sum']
print('-' * 34)
print(f"  {'TOTAL':<15}{tot:>14,.0f} ha")
print(f"  {'MCD64A1':<15}{burned_by_year[2024]:>14,.0f} ha  <- these should now agree")
Map

> ### ✓ Check your answer
>
> Now the total comes to **67,147 ha** against MCD64A1's **67,128 ha** — a 0.03% difference,
> which is just reprojection rounding. **The two methods now agree, so you can trust the
> severity breakdown:**
> 
> | Severity | Hectares |
> |---|---:|
> | Unburned | 3,702 |
> | Low | 18,897 |
> | Moderate-low | 25,356 |
> | Moderate-high | 16,680 |
> | High | 2,511 |
> 
> Only about **4%** of the burned area burned at high severity — typical for the surface fires that
> dominate Southeast Asian deciduous forest, as opposed to the crown fires you see in conifer systems.
> 
> Notice that 3,702 ha sit in the "Unburned" band even though MODIS mapped them as burn scar. The
> two products are 500 m and 10 m respectively, so a MODIS pixel counted as burned may be mostly
> unburnt ground. Disagreement at the edges is normal; disagreement by 24× is not.
> 
> **The habit to keep:** when two independent methods disagree by 24×, do not pick the prettier map.
> Find out which assumption broke.